# LDA Preprocessing Pipeline

## spaCy over NLTK 
- Speed: spaCY is faster than nltk for large scale during tokenisation
- Better lemmatisation: no need to set up POS stagging pipeline then wordness lemmatizer, word2vec can auto infer POS
- using en_core_web_lg model from spaC bc its faster
- ref: https://medium.com/data-science/practical-guide-to-topic-modeling-with-lda-05cd6b027bdf

In [54]:
import pandas as pd
import spacy
from sklearn.feature_extraction.text import CountVectorizer
from spacy.language import Language
from spacy.schemas import ConfigSchemaNlp
ConfigSchemaNlp.model_rebuild()
nlp = spacy.load('en_core_web_lg')
from sklearn.decomposition import LatentDirichletAllocation


# Stopword List

In [55]:
# ── Stop words ──────────────────────────────────────────────────────
FINANCE_STOPWORDS = {
    "herein", "thereof", "thereto", "therein", "hereby", "pursuant",
    "accordance", "aforementioned", "foregoing", "whereas", "whereby",
    "hereafter", "hereinafter",
    "form", "quarterly", "annual", "report", "filing", "fiscal",
    "quarter", "period", "ended", "condensed", "consolidated",
    "statements", "notes", "refer", "also", "following", "described",
    "set", "forth", "part", "item",
    "incorporated", "reincorporated", "headquartered", "mean", "means",
    "including", "included", "related", "thereto", "million", "thousand", "billion",
    "company", "corporation", "subsidiaries", "inc", "ltd", "limited",
    "platforms", "manufacturing",
    # generic financial terms that appear everywhere
    "year", "quarter", "net", "total", "company", "income", "sale",
    "stock", "share", "value", "loss", "cost", "rate", "increase",
    "decrease", "high", "low", "new", "old", "use", "provide",
    "result", "period", "fiscal", "annual", "month", "date",
    "future", "current", "prior", "continue", "compare", "impact",
    "significant", "general", "certain", "include", "consist",
    # SEC filing boilerplate
    "report", "form", "filing", "sec", "act", "note", "item",
    "statement", "financial", "accounting", "disclosure", "exhibit",
    "registrant", "amendment", "pursuant", "herein", "thereof",
    "whereas", "whereas", "thereto",
    # numbers/time that slipped through
    "january", "february", "march", "april", "may", "june",
    "july", "august", "september", "october", "november", "december",
    # spaCy lemmatization artifacts
    "datum",   # "data" → "datum"
    "agendum", # "agenda" → "agendum"
}

GEO_STOPWORDS = {
    "alabama", "alaska", "arizona", "arkansas", "california", "colorado",
    "connecticut", "delaware", "florida", "georgia", "hawaii", "idaho",
    "illinois", "indiana", "iowa", "kansas", "kentucky", "louisiana",
    "maine", "maryland", "massachusetts", "michigan", "minnesota",
    "mississippi", "missouri", "montana", "nebraska", "nevada",
    "new hampshire", "new jersey", "new mexico", "new york", "north carolina",
    "north dakota", "ohio", "oklahoma", "oregon", "pennsylvania",
    "rhode island", "south carolina", "south dakota", "tennessee", "texas",
    "utah", "vermont", "virginia", "washington", "west virginia",
    "wisconsin", "wyoming",

    "ca", "de", "tx", "ny", "wa", "nv", "fl", "ga", "il", "ma",
    "san francisco", "san jose", "santa clara", "santa", "clara",
    "palo alto", "menlo park", "mountain view", "sunnyvale", "cupertino",
    "redwood city", "fremont", "san diego", "los angeles", "seattle",
    "portland", "austin", "boston", "new york city", "new york",
    "chicago", "denver", "atlanta", "miami",
    "delaware", "nevada", "cayman islands", "bermuda", "ireland",
    "luxembourg", "singapore", "hong kong", "switzerland",
    "united states", "united kingdom", "china", "japan", "germany",
    "france", "india", "taiwan", "south korea", "israel", "canada",
    "australia", "netherlands", "sweden", "finland",
    "north america", "south america", "latin america", "europe",
    "asia pacific", "middle east", "apac", "emea", "americas",
}


ALL_STOP_WORDS = FINANCE_STOPWORDS | GEO_STOPWORDS

# Fix: use ALL STOP WORDS consistently in both places
nlp.Defaults.stop_words |= ALL_STOP_WORDS
for word in ALL_STOP_WORDS:        
    nlp.vocab[word].is_stop = True


# Tokenize & Lemmatize with spaCy

In [56]:
def preprocess_docs(texts: list[str], batch_size: int = 500) -> list[str]:
    """
    - Filters out: punctuation, whitespace, stop words, short tokens (<3 chars).
    - Returns one lemmatized string per document.
    """
    processed = []
 
    for doc in nlp.pipe(texts, disable=["parser", "ner"], batch_size=batch_size):
        tokens = [
            token.lemma_.lower()
            for token in doc
            if not token.is_space         # drop whitespace tokens
            and not token.is_stop          # drop stop words
            and len(token.lemma_) >= 3     # drop very short tokens
            and token.lemma_.isalpha()     # keep alphabetic tokens only
        ]
        processed.append(" ".join(tokens))
 
    return processed

# Vectorise with CountVectorizer

In [57]:
def build_count_matrix(processed_docs: list[str]):
    """
    Convert lemmatized documents into a sparse document-term count matrix.
    """
    vectorizer = CountVectorizer(
        min_df=10,                # token must appear in at least 10 documents -- remove rare words
        max_df=0.90,             # ignore tokens in >90% of documents -- remove too common words
        max_features=50_000,     # cap vocabulary size
        ngram_range=(1, 2),      # allow bi grams
    )
 
    dtm = vectorizer.fit_transform(processed_docs)
    return dtm, vectorizer

# Run Processing Pipeline

In [58]:
docs = pd.read_csv('../datasets/final/mda_shared_processed.csv')

In [ ]:
print("Step 1: Lemmatizing with spaCy...")
docs['processed_text'] = preprocess_docs(docs['clean_text'].tolist())

Step 1: Lemmatizing with spaCy...


In [ ]:
print("Step 2: Building document-term count matrix...")
dtm, vectorizer = build_count_matrix(docs['processed_text'])

Step 2: Building document-term count matrix...


In [ ]:
dtm ## USE THIS IN MODEL
print(dtm.shape)

(18177, 50000)


# LDA Model Example Usage

## PASS DTM INTO YOUR MODE TO USE

In [ ]:
lda = LatentDirichletAllocation(
    n_components=10,
    learning_method="online",
    total_samples=len(docs),
    batch_size=2000,
    learning_decay=0.7,
    learning_offset=100,
    random_state=42,
)
lda.fit(dtm)

,n_components,10
,doc_topic_prior,None
,topic_word_prior,None
,learning_method,'online'
,learning_decay,0.7
,learning_offset,100
,max_iter,10
,batch_size,2000
,evaluate_every,-1
,total_samples,18177
,perp_tol,0.1


In [ ]:
def get_top_words(lda, vectorizer, n_words=15):
    feature_names = vectorizer.get_feature_names_out()
    
    topics = {}
    for i, topic_dist in enumerate(lda.components_):
        top_indices = topic_dist.argsort()[::-1][:n_words]
        top_words = [feature_names[j] for j in top_indices]
        topics[f'topic_{i}'] = top_words
    
    return pd.DataFrame(topics)

# topic - word distribution (top 15 words per topic)
top_words_df = get_top_words(lda, vectorizer, n_words=15)
top_words_df.head(20)

,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9
0,company,tax,product,service,product,loan,credit,service,fund,service
1,share,value,development,product,service,interest,facility,sale,income,income
2,stock,asset,market,tax,market,rate,sale,subscription,service,asset
3,common,estimate,sale,sale,future,asset,service,operating,loss,sale
4,agreement,income,high,software,new,income,year,net,investment,stock
5,common stock,fair,manufacturing,income,technology,net,decrease,marketing,payment,interest
6,note,product,technology,year,company,value,company,income,tax,compensation
7,value,fair value,research,compare,time,loss,segment,consist,rate,term
8,warrant,service,design,decrease,stock,interest rate,net,development,net,general
9,interest,year,research development,high,security,bank,income,decrease,company,domain


In [ ]:
# Need this Aggregate back to company level 
doc_index = docs[['doc_id', 'company_name', 'year', 'quarter']].reset_index(drop=True)
doc_index.head() ## join back later

,doc_id,company_name,year,quarter
0,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,2020,Q1
1,1-800-PetMeds_10-K_2020-05-26,1-800-PetMeds,2020,Q2
2,1-800-PetMeds_10-Q_2020-08-03,1-800-PetMeds,2020,Q3
3,1-800-PetMeds_10-Q_2020-11-03,1-800-PetMeds,2020,Q4
4,1-800-PetMeds_10-Q_2021-01-26,1-800-PetMeds,2021,Q1


In [ ]:
doc_topic_matrix = lda.transform(dtm)    # shape: (n_docs, n_topics)
n_topics = doc_topic_matrix.shape[1]
topic_cols = [f'topic_{i+1}' for i in range(n_topics)]

docs_with_topics = docs[['doc_id', 'company_name', 'year', 'quarter']].copy()
docs_with_topics[topic_cols] = doc_topic_matrix

In [ ]:
company_year_topics = (
    docs_with_topics
    .groupby(['company_name', 'year'])[topic_cols]
    .mean()
    .reset_index()
)
company_year_topics.head()

,company_name,year,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,topic_10
0,1-800-PetMeds,2020,0.084161,0.549501,0.011721,0.001138,0.017805,0.000105,0.183104,0.014410,0.115471,0.022584
1,1-800-PetMeds,2021,0.044574,0.493469,0.088773,0.000089,0.017156,0.000089,0.209795,0.006220,0.117726,0.022110
2,1-800-PetMeds,2022,0.000061,0.512252,0.097351,0.000061,0.000061,0.000061,0.241327,0.016537,0.112676,0.019614
3,1-800-PetMeds,2023,0.065807,0.419554,0.000095,0.005829,0.354633,0.045692,0.038865,0.000095,0.069336,0.000095
4,1-800-PetMeds,2024,0.043275,0.463496,0.000099,0.018774,0.286995,0.036360,0.069908,0.000099,0.080895,0.000099


In [ ]:
company_quarter_topics = (
    docs_with_topics
    .groupby(['company_name', 'year', 'quarter'])[topic_cols]
    .mean()
    .reset_index()
)
company_quarter_topics.head()

,company_name,year,quarter,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,topic_10
0,1-800-PetMeds,2020,Q1,0.075893,0.585139,0.000112,0.003443,0.000112,0.000112,0.173937,0.000112,0.119145,0.041994
1,1-800-PetMeds,2020,Q2,0.119029,0.542302,0.012691,0.000071,0.000071,0.000071,0.132910,0.057290,0.118735,0.016829
2,1-800-PetMeds,2020,Q3,0.073268,0.545036,0.000123,0.000123,0.049171,0.000123,0.184795,0.000123,0.125615,0.021621
3,1-800-PetMeds,2020,Q4,0.068452,0.525525,0.033956,0.000916,0.021867,0.000114,0.240773,0.000114,0.098391,0.009893
4,1-800-PetMeds,2021,Q1,0.049071,0.502915,0.080604,0.000114,0.024858,0.000114,0.211064,0.000114,0.106743,0.024404
